# 14 — Past-only temporal motion across all 20 matches

**Aim:** Extend the successful single-match temporal pilot to the frozen 20-match sample, without introducing forward-looking data or treating displacement rates as player capabilities.

**Observation unit:** possession start. For each defender recorded in Notebook 05's canonical rows, compare their current position with **exact historical frames** at `t₀−5` and `t₀−10` (nominally 0.5 s and 1.0 s at 10 Hz). Keep the match and period key. Do not interpolate. Retain missingness and directly detected versus extrapolated endpoints.

**Inputs:** `outputs/P1_canonical_decisions_match_<match_id>_ROLES.jsonl` and original tracking JSONL in `data/matches`.  
**Outputs:** `outputs/P3_past_motion_match_<match_id>.jsonl` (local) and `outputs/P3_all_matches_motion_gate.json` (upload).

This is a **tracking-quality/retrospective-motion** step. It does not estimate defender sprint capacity, ball-pass velocity, interception reach, causal pressure effects, or carrying alternatives. No outcomes are used.


In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime,timezone
import json,math,statistics

ROOT=next((p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents)
           if (p/'data'/'matches').is_dir()),None)
if ROOT is None: raise FileNotFoundError('Launch notebook inside the SkillCorner repository.')
OUT=ROOT/'outputs'
MATCHES=ROOT/'data'/'matches'
MIDS=sorted(p.name for p in MATCHES.iterdir() if p.is_dir() and p.name.isdigit())
assert len(MIDS)==20,f'Expected 20 match folders, found {len(MIDS)}'
assert all((OUT/f'P1_canonical_decisions_match_{mid}_ROLES.jsonl').exists() for mid in MIDS)
print('Matches:',len(MIDS))


Matches: 20


## 1 — Explicit temporal and quality rules

We require an exact tracking frame at each possession start. Historical frames may be missing and are counted, never synthesized. A missing defender in a past frame gives a `null` rate. Endpoint-detected means **both endpoints** have `is_detected=True`; it does not establish that all intermediate frames were directly observed.

The spatial coordinates are used as published. We calculate average **net-displacement** over a window; this underestimates path length when players turn and is not instantaneous speed. The 15 m/s flag is an anomaly inspection marker only.


In [2]:
def integer(v):
    try: return int(v)
    except (TypeError,ValueError): return None

def xy(p):
    if not isinstance(p,dict):return None
    try:
        a,b=float(p['x']),float(p['y'])
        return (a,b) if math.isfinite(a) and math.isfinite(b) else None
    except (KeyError,TypeError,ValueError):return None

def players(frame):
    return {str(p['player_id']):p for p in (frame.get('player_data') or [])
            if p.get('player_id') is not None}

def motion(p0,ph,gap):
    a,b=xy(p0),xy(ph)
    if a is None or b is None:return None
    return math.hypot(a[0]-b[0],a[1]-b[1])/(gap/10.0)

def dist_summary(v):
    if not v:return {'n':0,'median':None,'p95':None,'max':None}
    s=sorted(v)
    return {'n':len(s),'median':statistics.median(s),
            'p95':s[min(len(s)-1,int(0.95*(len(s)-1)))],'max':s[-1]}

def one_match(mid):
    canonical=OUT/f'P1_canonical_decisions_match_{mid}_ROLES.jsonl'
    decisions=[json.loads(line) for line in canonical.open(encoding='utf-8') if line.strip()]
    wanted={(int(r['period']),int(r['frame'])-h)
            for r in decisions for h in (0,5,10) if int(r['frame'])>=h}
    frames={}
    invalid_keys=0
    with (MATCHES/mid/f'{mid}_tracking_extrapolated.jsonl').open(encoding='utf-8') as stream:
        for line_number,line in enumerate(stream,1):
            rec=json.loads(line)
            p,f=integer(rec.get('period')),integer(rec.get('frame'))
            if p is None or f is None:
                invalid_keys+=1
                continue
            k=p,f
            if k in wanted:
                if k in frames:
                    raise AssertionError(f'{mid}: duplicate requested key {k}')
                frames[k]=rec
    counts=Counter()
    all_halfsec=[]
    detected_halfsec=[]
    ball_halfsec=[]
    out=OUT/f'P3_past_motion_match_{mid}.jsonl'
    with out.open('w',encoding='utf-8') as dest:
        for row in decisions:
            p,t=int(row['period']),int(row['frame'])
            now=frames.get((p,t))
            if now is None:
                raise AssertionError(f'{mid}: missing required present frame {(p,t)}')
            current=players(now)
            prior_frames={h:frames.get((p,t-h)) if t>=h else None for h in (5,10)}
            prior_players={h:players(z) if z is not None else {} for h,z in prior_frames.items()}
            defenders=[]
            ids=set()
            for quality in ('detected','extrapolated'):
                for d in row.get(f'opponents_{quality}',[]):
                    pid=str(d['player_id'])
                    if pid in ids:raise AssertionError(f'{mid}: duplicate defender {pid} at {(p,t)}')
                    ids.add(pid)
                    p0=current.get(pid)
                    if p0 is None:
                        raise AssertionError(f'{mid}: canonical defender missing from tracking {(p,t,pid)}')
                    entry={'player_id':pid,'t0_quality':quality}
                    for h in (5,10):
                        ph=prior_players[h].get(pid)
                        val=motion(p0,ph,h)
                        both=(p0.get('is_detected') is True and
                              ph is not None and ph.get('is_detected') is True)
                        entry[f'past_{h}_avg_net_displacement_mps']=val
                        entry[f'past_{h}_endpoints_detected']=bool(both)
                        counts[f'defender_{h}_finite']+=val is not None
                        counts[f'defender_{h}_detected_endpoints']+=val is not None and both
                        if h==5 and val is not None:
                            all_halfsec.append(val)
                            if both: detected_halfsec.append(val)
                            counts['defender_halfsec_gt_15mps']+=val>15
                    defenders.append(entry)
            counts['defender_records']+=len(defenders)
            b0=now.get('ball_data') or {}
            ball_rates={}
            carrier_rates={}
            for h in (5,10):
                ph=prior_frames[h]
                counts[f'past_{h}_frame_present']+=ph is not None
                bprev=(ph.get('ball_data') or {}) if ph is not None else None
                bv=motion(b0,bprev,h)
                ball_rates[str(h)]=bv
                counts[f'ball_{h}_finite']+=bv is not None
                if h==5 and bv is not None:ball_halfsec.append(bv)
                carrier=motion(current.get(str(row['carrier_player_id'])),
                               prior_players[h].get(str(row['carrier_player_id'])),h)
                carrier_rates[str(h)]=carrier
                counts[f'carrier_{h}_finite']+=carrier is not None
            record={'match_id':mid,'period':p,'frame':t,
                    'possession_event_id':row['possession_event_id'],
                    'carrier_player_id':row['carrier_player_id'],
                    'past_defender_motion':defenders,
                    'past_carrier_avg_net_displacement_mps':carrier_rates,
                    'past_ball_avg_net_displacement_mps':ball_rates,
                    'past_frame_available':{str(h):prior_frames[h] is not None for h in (5,10)},
                    'interpretation':'past_only_net_displacement_not_interception_capacity'}
            dest.write(json.dumps(record,separators=(',',':'))+'\n')
            counts['decisions']+=1
    if counts['decisions']!=len(decisions):
        raise AssertionError(f'{mid}: decision count mismatch')
    return {'match_id':mid,'decisions':len(decisions),
            'unique_requested_keys':len(wanted),'unique_found_keys':len(frames),
            'unavailable_requested_keys':len(wanted)-len(frames),
            'invalid_key_tracking_records':invalid_keys,'counts':dict(counts),
            'defender_halfsec_all':dist_summary(all_halfsec),
            'defender_halfsec_both_endpoints_detected':dist_summary(detected_halfsec),
            'ball_halfsec_historical':dist_summary(ball_halfsec)}


## 2 — Cross-match execution

Each source match is read independently. Failed required frames and duplicate keys stop execution. Missing *historical* frames are reported as missing; they do not invalidate correctly linked decision-time observations.


In [3]:
results=[]
for mid in MIDS:
    result=one_match(mid)
    results.append(result)
    c=result['counts']
    print(mid,'decisions:',result['decisions'],
          'historical frames (−5, −10):',
          c.get('past_5_frame_present',0),c.get('past_10_frame_present',0),
          'defender finite (−5):',c.get('defender_5_finite',0))


1874553 decisions: 959 historical frames (−5, −10): 959 959 defender finite (−5): 10395
1886347 decisions: 999 historical frames (−5, −10): 999 999 defender finite (−5): 10901
1899585 decisions: 929 historical frames (−5, −10): 927 927 defender finite (−5): 10076
1925299 decisions: 1062 historical frames (−5, −10): 1062 1062 defender finite (−5): 11649
1927964 decisions: 1140 historical frames (−5, −10): 1140 1140 defender finite (−5): 12353
1953632 decisions: 979 historical frames (−5, −10): 979 979 defender finite (−5): 10571
1959846 decisions: 946 historical frames (−5, −10): 945 945 defender finite (−5): 10252
1986691 decisions: 915 historical frames (−5, −10): 915 915 defender finite (−5): 9900
1996435 decisions: 1005 historical frames (−5, −10): 1005 1005 defender finite (−5): 10879
1996436 decisions: 786 historical frames (−5, −10): 786 786 defender finite (−5): 8580
2006229 decisions: 979 historical frames (−5, −10): 979 978 defender finite (−5): 10670
2006363 decisions: 976 hi

## 3 — Aggregated gate

These reported totals are *measurements of data support and descriptive retrospective motion*. They are not a parameter-calibration set for a speed or interception model. Compare quality and distributions **per match**, avoiding pseudo-independent inference on defender observations.


In [4]:
counts=Counter()
for r in results:counts.update(r['counts'])
summary={'created_utc':datetime.now(timezone.utc).isoformat(),
 'purpose':'Past-only 0.5s and 1s retrospective tracking-motion coverage; descriptive only',
 'n_matches':len(results),
 'decisions':sum(r['decisions'] for r in results),
 'unique_requested_keys':sum(r['unique_requested_keys'] for r in results),
 'unique_found_keys':sum(r['unique_found_keys'] for r in results),
 'unavailable_requested_keys':sum(r['unavailable_requested_keys'] for r in results),
 'counts':dict(counts),
 'matches':results,
 'limitations':[
   'Exact historical frame keys within the same period; no future frames',
   'Net displacement per second is not instantaneous speed, path distance, acceleration, or sprint capacity',
   'Endpoint detection does not certify the complete historical trajectory',
   'Extrapolated coordinates can distort motion',
   'Historical ball movement is not prospective pass-travel velocity',
   '15m/s is a diagnostic flag only; no automatic exclusions',
   'This notebook does not construct interception probabilities or feasible passing/carrying options',
   'Defenders within a possession and observations within a match are not statistically independent']}
assert summary['n_matches']==20
assert summary['decisions']==18792
assert counts['decisions']==18792
path=OUT/'P3_all_matches_motion_gate.json'
path.write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps({k:summary[k] for k in ('n_matches','decisions','unique_requested_keys','unique_found_keys','unavailable_requested_keys','counts')},indent=2))
print('Upload:',path)


{
  "n_matches": 20,
  "decisions": 18792,
  "unique_requested_keys": 55901,
  "unique_found_keys": 55873,
  "unavailable_requested_keys": 28,
  "counts": {
    "defender_5_finite": 204127,
    "defender_5_detected_endpoints": 130769,
    "defender_halfsec_gt_15mps": 1,
    "defender_10_finite": 203170,
    "defender_10_detected_endpoints": 125608,
    "defender_records": 206338,
    "past_5_frame_present": 18778,
    "ball_5_finite": 18557,
    "carrier_5_finite": 18556,
    "past_10_frame_present": 18776,
    "ball_10_finite": 18470,
    "carrier_10_finite": 18469,
    "decisions": 18792
  }
}
Upload: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P3_all_matches_motion_gate.json
